# WriteSAE quickstart

Load the released L9 H4 WriteSAE checkpoint and run a shape smoke test.


In [ ]:
import os
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

from core.sae import build_sae_from_config, infer_matrix_dims

HF_REPO = os.environ.get("WRITESAE_HF_REPO", "JackYoung27/writesae-ckpts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}")


In [ ]:
root = Path(snapshot_download(
    repo_id=HF_REPO,
    allow_patterns=["writesae/qwen0p8b/L9_H4/*"],
    cache_dir=".hf_cache",
))
ckpt_path = root / "writesae/qwen0p8b/L9_H4/best.pt"
ckpt = torch.load(ckpt_path, weights_only=False, map_location="cpu")
print(ckpt_path)
print(ckpt.get("config", {}))
print(ckpt.get("val_mse", ckpt.get("best_val_mse")))


In [ ]:
if "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
    cfg = ckpt.get("config", {})
    sae = build_sae_from_config(cfg, state_dict=state_dict)
    sae.load_state_dict(state_dict)
    d_k, d_v = infer_matrix_dims(cfg, state_dict)
else:
    sae = ckpt["sae"]
    cfg = ckpt.get("config", {})
    d_k = int(cfg.get("d_k", getattr(sae, "d_k", 128)))
    d_v = int(cfg.get("d_v", getattr(sae, "d_v", 128)))

sae = sae.to(DEVICE).eval()
x = torch.zeros(1, d_k, d_v, device=DEVICE)
with torch.no_grad():
    out = sae(x)

print({
    "input": tuple(x.shape),
    "reconstruction": tuple(out.reconstruction.shape),
    "coefficients": tuple(out.coefficients.shape),
    "mse": float(out.mse.detach().cpu()),
})
